# Ch6 Tokenizer 教案

**课程名称：** Tokenizer：数据预处理的艺术——把文本变成模型能理解的数字

**预计总时长：** 75-85 分钟

**源文件：** `Ch6_Tokenizer/Ch6_Tokenizer.ipynb`（共 24 个 Cell，Cell 0-23）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00-05:00 | 开场与环境准备 | Cell 0-4 | 5 分钟 |
| 05:00-15:00 | 为什么需要 Tokenizer + 三种分词策略对比 | Cell 5-6 | 10 分钟 |
| 15:00-35:00 | BPE 算法：从头实现 | Cell 7-9 | 20 分钟 |
| 35:00-40:00 | 休息 + 回顾 | -- | 5 分钟 |
| 40:00-50:00 | tiktoken 实战：工业级 Tokenizer | Cell 10-13 | 10 分钟 |
| 50:00-65:00 | Tokenizer 导致的经典 Bug | Cell 14-17 | 15 分钟 |
| 65:00-72:00 | 可视化词表覆盖 + 总结 | Cell 18-22 | 7 分钟 |
| 72:00-85:00 | 练习：实现 BPE encode | Cell 21, 23 | 13 分钟 |

---

## 课前准备

- [ ] 确认 Python 3.11+ 环境可用
- [ ] 确认 `tiktoken` 已安装（`pip install tiktoken`）
- [ ] 确认 `numpy`、`matplotlib` 可用
- [ ] 确认中文字体设置正确（Microsoft YaHei / SimHei）
- [ ] 提前运行一遍全部 Cell，确认无报错
- [ ] 准备白板/画板用于画 BPE 合并过程
- [ ] 打开 Andrej Karpathy 的 "Let's build the GPT Tokenizer" YouTube 视频页面备用
- [ ] 准备好一个中英文混合的 ChatGPT 对话截图，用于演示 Token 消耗差异

---

## 第一段：开场与环境准备（Cell 0-4）

📍 运行 Cell 0-3（Markdown：标题、学习路线、前置知识）、Cell 4（代码：环境准备）

⏱ 时间分配：5 分钟

🎯 本段目标
- 建立学习动机：我们一直在说"Token 消耗"，到底消耗的是什么？
- 确认环境就绪（numpy、matplotlib、tiktoken）
- 让学生对 Tokenizer 在 LLM 管线中的位置有全局感

🗣 讲课话术

> 大家好！今天我们来聊一个你每天都在用，但可能从来没仔细想过的东西——Tokenizer。
>
> 你们用 ChatGPT 的时候，有没有注意过 API 返回里有个"token 用量"？比如一句"Hello World"消耗了 2 个 token，而"你好世界"消耗了 4 个 token。同样四个字，为什么中文贵一倍？
>
> 今天我们就来彻底搞清楚这件事。Tokenizer 干的事情说白了就一件：把人类的文字变成模型能吃进去的数字。源 notebook Cell 2 那个图画得很清楚：`"Hello World" → [15496, 2159] → 神经网络 → [1312] → "I"`。
>
> 我们先把环境跑起来。运行 Cell 4，看到"环境准备完成！"就行。

👀 输出要点
- Cell 4 应输出：`环境准备完成！`
- 如果 tiktoken 未安装，取消注释 Cell 4 第一行 `!pip install tiktoken`

❓ 预判问题

Q: Tokenizer 和 Embedding 有什么关系？
A: Tokenizer 把文本变成 Token ID（整数），Embedding 把 Token ID 变成向量。Tokenizer 是第一步，Embedding 是第二步。可以回顾 Ch2 的内容。

Q: 为什么不直接把每个字符映射成数字？
A: 可以，但序列会非常长。"Hello World" 字符级要 11 个 token，子词级只要 2 个。Attention 的计算量是序列长度的平方，所以更短的序列意味着更少的计算。我们马上会详细对比。

➡️ 转场

> 环境没问题。接下来我们先回答一个最基本的问题：文本有那么多种切法，到底哪种最好？

---

## 第二段：为什么需要 Tokenizer + 三种分词策略对比（Cell 5-6）

📍 运行 Cell 5（Markdown：分词策略理论）、Cell 6（代码：三种分词策略演示）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解字符级、词级、子词级三种分词策略的优缺点
- 理解词表大小 V 是一个参数-计算权衡
- 直观看到同一句话在不同策略下的 token 数差异

🗣 讲课话术

> 我打个比方。假设你要给外星人传一本中文小说，但你们之间只能传数字。
>
> 方案一：字符级。给每个汉字、标点编个号，词表大概几千个。但"机器学习"要传 4 个数字，一本书下来传的数字串特别长。
>
> 方案二：词级。给每个词编号，"机器学习"就一个数字搞定。但你得准备一本巨大的词典——几十万个词，而且遇到新词（比如"ChatGPT"）直接抓瞎，这就是所谓的 OOV（Out-of-Vocabulary）问题。
>
> 方案三：子词级。把"unhappiness"拆成"un" + "happiness"，高频词保持完整，低频词拆成有意义的片段。这就是 BPE。
>
> 现在运行 Cell 6 看看真实的对比。同一句 `"Hello, I'm learning machine learning!"`——字符级切出 **37 个 token**，词级只有 **5 个**（但注意标点被粘在词上了），子词级是 **8 个**。
>
> Cell 5 的理论部分还有一个很重要的权衡分析：Embedding 矩阵大小是 V 乘以 d_model。如果 V = 100,000，d_model = 4096，光 Embedding 就有 4 亿参数！但词表大了序列就短了，Attention 的 O(T^2) 计算量就省了。现代 LLM 通常选 32K 到 150K 的词表大小，在两者之间找平衡。

👀 输出要点
- Cell 6 输出：
  - 字符级分词 (37 tokens): `['H', 'e', 'l', 'l', 'o', ',', ' ', 'I', "'", 'm', ...]`
  - 词级分词 (5 tokens): `['Hello,', "I'm", 'learning', 'machine', 'learning!']`
  - 子词级分词 (8 tokens): `['Hello', ',', ' I', "'m", ' learning', ' machine', ' learning', '!']`
- 重点强调：37 vs 5 vs 8 这三个数字，让学生对"序列长度差异"有直观感受

❓ 预判问题

Q: 子词级为什么有些 token 前面带空格（比如 ` learning`）？
A: 这是 BPE 的特点——空格被编码到 token 里面，而不是作为独立 token。这样模型能区分句首的"learning"和句中的" learning"。这个设计后面在"坑点分析"部分会详细讲。

Q: 词表大小怎么选？有公式吗？
A: 没有精确公式，是经验性的。GPT-2 用 50,257，GPT-4 用约 100K（cl100k_base），LLaMA 用 32K。一般训练数据越大、越多语言，词表可以越大。

Q: 信息论视角的"压缩率"怎么理解？
A: Cell 5 提到优秀英文 Tokenizer 能达到 3-5 bytes/token——意思是平均每个 token 代表 3-5 个字符。压缩率越高，说明 Tokenizer 越会"发现"文本中的规律性模式。

➡️ 转场

> 好，子词级分词是最佳方案。但它到底怎么学会把"unhappy"拆成"un"+"happy"的呢？答案就是 BPE 算法。接下来我们从零实现它。

---

## 第三段：BPE 算法——从头实现（Cell 7-9）

📍 运行 Cell 7（Markdown：BPE 理论）、Cell 8（代码：get_stats + merge_vocab + 初始词表）、Cell 9（代码：BPE 训练过程 + 合并频率可视化）

⏱ 时间分配：20 分钟（理论 5 分钟 + Cell 8 代码讲解 5 分钟 + Cell 9 训练过程 10 分钟）

🎯 本段目标
- 理解 BPE 的核心循环：统计频率 → 找最高频对 → 合并 → 重复
- 能读懂 `get_stats` 和 `merge_vocab` 两个函数
- 通过 10 次合并的完整过程，直观理解词表是怎样"长大"的

🗣 讲课话术

> BPE 的思路其实特别朴素——贪心算法。我用一个生活中的例子来类比：
>
> 想象你在发短信，为了省字数，你和朋友约定缩写。你们发现"哈哈"出现得最多，于是约定用"H"代替"哈哈"。然后"不是"也很常见，用"B"代替。这就是 BPE 的核心思想——统计最常见的相邻组合，然后把它们合并成一个新符号。
>
> 先看 Cell 7 的形式化描述。算法输入是语料库和目标词表大小。初始词表就是所有字符（或字节级 BPE 的 256 个字节）。然后不断循环：统计所有相邻 token 对的频率 → 合并频率最高的对 → 直到达到目标词表大小。
>
> 顺便提一下时间复杂度：每轮合并要扫描整个语料 O(n)，总共 V 轮，所以是 O(n * V)。大规模语料上会用优先队列加速。
>
> 好，现在看代码。运行 Cell 8。我们的训练语料是 `"low low low low low lower lower newest newest newest newest newest newest widest widest widest"`。
>
> `get_stats` 函数做的事情很简单：遍历词表里每个词，统计相邻 token 对的出现次数。`merge_vocab` 就是把指定的字符对在整个词表里合并起来。
>
> 看初始词表的输出：
> - `'l o w </w>'`: 5 次（"low" 出现了 5 次）
> - `'l o w e r </w>'`: 2 次
> - `'n e w e s t </w>'`: 6 次
> - `'w i d e s t </w>'`: 3 次
>
> 每个词都已经按字符拆开了，末尾加了 `</w>` 作为词结束标记。
>
> 现在运行 Cell 9，看 10 次合并的完整过程。这是整节课最关键的地方，大家仔细看：
>
> **第 1 次合并**：`('e', 's')` 频率 9——为什么是 9？因为 "newest" 里有 es 出现 6 次，"widest" 里有 es 出现 3 次，6+3=9。合并后 `e s` 变成 `es`。
>
> **第 2 次合并**：`('es', 't')` 频率 9——刚合并出的 `es` 和 `t` 又是最高频对！合并成 `est`。
>
> **第 3 次合并**：`('est', '</w>')` 频率 9——`est` 总在词尾，所以和结束标记合并。
>
> 看到规律了吗？前三步就把 "est" 这个常见后缀学出来了！这就是 BPE 的美妙之处——它自动发现了英语中有意义的子词。
>
> **第 4-5 次合并**：`('l', 'o')` 频率 7，然后 `('lo', 'w')` 频率 7——把 "low" 也学出来了。
>
> **第 6-8 次**：学出了 "new" 然后和 "est</w>" 合并成 "newest</w>"。
>
> 最终词表只有 4 个条目：`low</w>`、`low e r </w>`、`newest</w>`、`wi d est</w>`。高频词如 "low" 和 "newest" 被完整保留，低频词如 "lower" 和 "widest" 仍有部分拆分。
>
> 下面那个柱状图也很直观——前 3 次合并的频率都是 9，因为它们都来自 "est" 这个后缀。

👀 输出要点
- Cell 8 输出初始词表 4 个条目，频率分别为 5、2、6、3
- Cell 9 输出 10 次合并的完整过程：
  - 第 1-3 次：`(e,s)→es→est→est</w>`，频率都是 9
  - 第 4-5 次：`(l,o)→lo→low`，频率都是 7
  - 第 6-8 次：`(n,e)→ne→new→newest</w>`，频率 6
  - 第 9 次：`(low,</w>)→low</w>`，频率 5
  - 第 10 次：`(w,i)→wi`，频率 3
- Cell 9 柱状图：BPE 合并频率 Top 10，前 3 个最高（9），依次递减
- 最终词表：`low</w>`:5, `low e r </w>`:2, `newest</w>`:6, `wi d est</w>`:3

❓ 预判问题

Q: `</w>` 结束标记有什么用？
A: 用来区分"词内"和"词尾"的 token。比如 "low" 作为完整词是 `low</w>`，作为 "lower" 的前缀是 `low`（没有结束标记）。这样 decode 的时候能正确恢复空格。

Q: 如果两个 pair 频率一样怎么办？
A: 代码里用的是 `max(pairs, key=pairs.get)`，Python 的 max 会返回第一个最大值。实际实现中通常有 tie-breaking 策略（比如按字典序），但对最终结果影响不大。

Q: BPE 和 Huffman 编码有什么区别？
A: Cell 7 有对比。两者都是贪心策略，本质上都在追求压缩效率。区别在于 BPE 是自底向上合并，Huffman 是基于完整频率表的最优构造。BPE 的优势是产出的 token 序列可以直接作为模型输入。

Q: 为什么现代 LLM 用"字节级" BPE 而不是"字符级"？
A: Cell 7 解释了——Unicode 有十几万个字符，而字节只有 256 个。字节级 BPE 的基础词表极小（256），且能编码任何文本（中文、emoji 都行），永远不会出现未登录词。GPT-2 开始采用这种方式。

➡️ 转场

> 我们刚才手写了一个 BPE 训练器。但工业级的 Tokenizer 长什么样？接下来我们用 OpenAI 开源的 tiktoken 来体验。先休息 5 分钟。

---

## 休息 + 回顾（第 35-40 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 分词有三种策略：字符级（词表小、序列长）、词级（序列短、OOV 问题）、子词级 BPE（最佳平衡），同一句话分别切出 37、5、8 个 token。
2. BPE 算法的核心是贪心合并：每轮统计最高频的相邻 token 对并合并，10 次合并就自动学出了 "est"、"low"、"new" 等有意义的子词。
3. 词表大小是参数量和计算量之间的权衡——V=100K、d_model=4096 时光 Embedding 就有 4 亿参数。

**下一段预告：**

> 接下来我们用 OpenAI 的 tiktoken 看看真实世界的 Tokenizer，特别是中英文效率差异——为什么中文调 API 更贵？

---

## 第四段：tiktoken 实战——工业级 Tokenizer（Cell 10-13）

📍 运行 Cell 10（Markdown：tiktoken 标题）、Cell 11（代码：tiktoken 基本用法）、Cell 12（pip install 备用）、Cell 13（代码：多语言效率对比）

⏱ 时间分配：10 分钟

🎯 本段目标
- 学会使用 tiktoken 对文本进行编码和解码
- 理解每个 token 对应的实际文本片段
- 直观感受中英文分词效率的巨大差异

🗣 讲课话术

> 我们刚才自己实现的 BPE 是教学版，实际 GPT-4 用的 Tokenizer 是 `cl100k_base`，词表有大约 10 万个 token。OpenAI 把它开源了，叫 tiktoken。
>
> 运行 Cell 11。输入是 `"Hello, 你好！I'm learning about tokenizers."`，一共被切成 **13 个 token**。
>
> 大家仔细看每个 token 的解码结果：
> - `9906: 'Hello'`——英文常见词完整保留
> - `11: ','`——标点单独一个 token
> - `220: ' '`——空格也是一个独立 token
> - `57668: '你'`、`53901: '好'`——注意！每个中文字都是单独的 token
> - `6447: '！'`——中文感叹号也占一个 token
> - `6975: ' learning'`——注意前面有空格，作为一个整体
> - `4037: ' token'` + `12509: 'izers'`——"tokenizers" 被拆成了两个子词
>
> 这里有个很有趣的现象：英文 "Hello" 一个 token，中文 "你好" 要两个 token。为什么？因为 GPT 的训练数据以英文为主，英文常见词被合并成了完整的 token，而中文字符在训练数据中出现频率相对低，没有被充分合并。
>
> 现在运行 Cell 13 看更系统的对比。三种语言的效率：
> - **English**：56 字符，9 个 token，效率 **6.22** 字符/token
> - **Chinese**：15 字符，**16** 个 token，效率 **0.94** 字符/token
> - **Code**：35 字符，9 个 token，效率 **3.89** 字符/token
>
> 看到了吗？中文的效率不到 1！意味着平均一个中文字符需要 1 个多 token 来表示。而英文一个 token 能代表 6 个多字符。这就是为什么用中文调 API 比英文贵——你付的钱是按 token 计费的，相同语义的内容中文消耗的 token 数量是英文的好几倍。
>
> 不过好消息是，新一代模型（如 Claude 3、GPT-4o）在训练数据中增加了中文比例，中文效率有所改善。

👀 输出要点
- Cell 11 输出：13 个 token，逐个解码展示
  - 关键对比：`'Hello'` 1 个 token，`'你'` 和 `'好'` 各 1 个 token
  - `'tokenizers'` 被拆成 `'token'` + `'izers'`
- Cell 13 输出：
  - English: 56 字符 / 9 tokens = 6.22 字符/token
  - Chinese: 15 字符 / 16 tokens = 0.94 字符/token
  - Code: 35 字符 / 9 tokens = 3.89 字符/token
- 重点强调：中文 15 个字符竟然需要 16 个 token（比字符数还多！）

❓ 预判问题

Q: 为什么中文 15 个字符变成 16 个 token，比字符还多？
A: 因为中文标点"。"在 UTF-8 编码中占 3 个字节，在字节级 BPE 中可能被拆分成多个 token。而且这里的"字符"是 Python 的 `len()`，一个中文字算 1 个字符，但底层 UTF-8 编码占 3 个字节。

Q: `cl100k_base` 里的 100k 是什么意思？
A: 词表大约有 100,000 个 token，所以叫 cl100k（100 thousand）。"cl" 可能是 "codebook language" 的缩写，"base" 表示基础版本。

Q: 代码的效率为什么比中文高？
A: GPT 的训练数据中有大量代码（GitHub 数据），所以 `def`、`return`、缩进等代码模式被充分合并成了高效的 token。

➡️ 转场

> tiktoken 看起来很智能，但 Tokenizer 并不完美。接下来我们看几个它会导致的经典 Bug——这些坑你在使用 LLM 时一定会遇到。

---

## 第五段：Tokenizer 导致的经典 Bug（Cell 14-17）

📍 运行 Cell 14（Markdown：四大陷阱理论）、Cell 15（代码：字符串反转 Bug）、Cell 16（代码：数字处理 Bug）、Cell 17（代码：空格敏感 Bug）

⏱ 时间分配：15 分钟

🎯 本段目标
- 理解 LLM 操作的是 token 而非字符，这导致字符串反转等任务失败
- 理解数字被不一致地切分，导致数学计算不稳定
- 理解空格和大小写如何影响分词结果
- 了解 SolidGoldMagikarp 等"幽灵 token"现象

🗣 讲课话术

> 接下来是本章最有意思的部分——Tokenizer 的坑。这些坑不是 bug，而是设计上的必然结果。理解它们对调试 LLM 应用至关重要。
>
> **Bug 1：字符串反转。** 运行 Cell 15。让我们看 `"lollipop"` 这个词。tiktoken 把它切成了 `['l', 'ollipop']`——对，两个 token，token ID 分别是 75 和 90644。
>
> 如果 LLM 按 token 反转，得到的是 `['ollipop', 'l']`，也就是 `"ollipopl"`。但正确答案应该是 `"popillol"`！
>
> 为什么？因为 LLM 根本看不到单个字符！它看到的最小单位就是 token。让它反转字符串，就好比让一个只能看到"词"的人去反转"字母"——粒度不对。
>
> 这就是为什么你让 ChatGPT 数"strawberry"里有几个 r，它经常数错——它压根看不到单个字母。
>
> **Bug 2：数字处理。** 运行 Cell 16。看看数字是怎么被切分的：
> - `"123"` → 1 个 token：`['123']`
> - `"1234"` → 2 个 token：`['123', '4']`
> - `"12345"` → 2 个 token：`['123', '45']`
> - `"123456789"` → 3 个 token：`['123', '456', '789']`
>
> 问题来了——如果模型要算 123456789 + 1，它看到的是三个独立的 token ['123', '456', '789']，进位信息要跨 token 传递，这非常困难。这也是 LLM 做数学不稳定的重要原因之一。
>
> **Bug 3：空格敏感。** 运行 Cell 17。同一个词 `hello`：
> - `'hello'` → token ID **15339**
> - `' hello'`（带前导空格）→ token ID **24748**
> - `'  hello'`（两个空格）→ 2 个 token：**220** + **24748**
> - `'Hello'`（大写）→ token ID **9906**
>
> 四种写法，四个完全不同的 token（或 token 组合）！模型看到的是不同的数字，需要靠训练来学习"这些其实是同一个词"。
>
> Cell 14 还提到了一个很有趣的现象——**SolidGoldMagikarp**。这是 Reddit 上一个用户名，在 BPE 训练时被合并成了一个 token，但在实际训练数据中几乎没出现过。它的 Embedding 向量基本是随机的，如果你强迫模型生成这个 token，行为完全不可预测。这类"幽灵 token"揭示了词表构建和训练数据之间的不匹配问题。

👀 输出要点
- Cell 15 输出：
  - `lollipop` → tokens [75, 90644]，即 `['l', 'ollipop']`
  - 按 token 反转：`['ollipop', 'l']`
  - 正确反转：`popillol`
- Cell 16 输出：
  - `123` → 1 token，`1234` → 2 tokens，`12345` → 2 tokens，`123456789` → 3 tokens
  - 注意 `1234` 的切法是 `['123', '4']` 而不是 `['12', '34']`——切分方式取决于训练数据中的频率
- Cell 17 输出：
  - `'hello'` → 15339，`' hello'` → 24748，`'Hello'` → 9906
  - 三种写法 → 三个不同的 token ID

❓ 预判问题

Q: 那 LLM 怎么做字符串反转？
A: 现代 LLM（如 GPT-4、Claude）通常是靠 Chain-of-Thought 把词拆成字符再处理，或者使用代码执行（Code Interpreter）来完成。Tokenizer 层面无法直接解决这个问题。

Q: 有没有解决数字切分问题的方案？
A: 有几种思路：(1) 每个数位一个 token（一些模型采用）；(2) 用工具调用计算器做数学；(3) 在 prompt 中让模型逐位计算。但根本性的解决需要改 Tokenizer 设计。

Q: SolidGoldMagikarp 现象在实际使用中会遇到吗？
A: 正常使用不会——这类 token 几乎不会在自然文本中出现。但对于安全研究者来说，这是一种可能的攻击向量（adversarial prompt）。

➡️ 转场

> 这些 Bug 都是 Tokenizer 设计的必然结果。最后我们来看一个词表的全局视图，然后总结全章。

---

## 第六段：可视化词表覆盖 + 总结（Cell 18-22）

📍 运行 Cell 18（Markdown：标题）、Cell 19（代码：词表构成饼图 + 语言效率条形图）、浏览 Cell 20-22（总结、Extra、下一步）

⏱ 时间分配：7 分钟

🎯 本段目标
- 通过可视化理解典型 LLM 词表的构成
- 回顾全章核心概念
- 为下一章 Pretraining 做铺垫

🗣 讲课话术

> 运行 Cell 19 看两张图。
>
> 左边的饼图展示了一个典型 GPT-4 风格词表的近似构成：**45%** 是英文词，**20%** 是代码 token，**15%** 是中文字符，**10%** 标点，**5%** 数字，**5%** 特殊 token。注意这些是近似值，用于说明趋势。
>
> 右边的条形图更直观——不同语言的分词效率。英文 **5.2** 字符/token 最高，西班牙语 **4.8**，德语 **4.5**，代码 **3.8**，日语 **2.0**，中文 **1.8**。英语效率是中文的将近 3 倍！
>
> 这意味着什么？如果你有一个 4K token 的上下文窗口：
> - 英文大约能放 4000 * 5.2 ≈ 20,000 个字符
> - 中文只能放 4000 * 1.8 ≈ 7,200 个字符
> - 中文能装的信息量不到英文的一半！
>
> 现在看 Cell 20 的总结。那个 ASCII 流程图把 Tokenizer 的全流程画得很清楚：`Raw Text → Byte → BPE Merges → Token IDs → Embedding 查表`。BPE 训练则是：`初始字符表 → 统计最频繁相邻字符对 → 合并为新 token → 重复至目标 V`。
>
> Cell 20 还有三道面试常考题，大家务必看：
> - Q1：为什么用子词分词？——字符级序列太长 O(T^2)，词级 OOV，子词级最佳平衡
> - Q2：BPE vs WordPiece vs SentencePiece 的区别
> - Q3：Tokenizer 导致的经典 Bug

👀 输出要点
- Cell 19 左图：饼图，English words 45%, Code tokens 20%, Chinese chars 15%
- Cell 19 右图：水平条形图，English 5.2 最高，Chinese 1.8 较低
- Cell 19 底部文字："注意：以上数字为近似/示意值"、"英语效率最高"、"中文每个字符约占 1-2 个 token"
- Cell 20：核心概念图谱（ASCII 流程图）、关键公式速查表、三道面试题

❓ 预判问题

Q: BPE 和 WordPiece 具体有什么区别？
A: Cell 20 的面试题 Q2 有详细对比。BPE 选频率最高的 pair，WordPiece（BERT 用的）选合并后使语言模型似然增加最大的 pair。SentencePiece（LLaMA 用的）不预先按空格分词，支持无空格语言。

Q: 为什么不能用 GPT-4 的 Tokenizer 解码 LLaMA 的输出？
A: 因为不同模型的词表完全不同！Token ID 42 在 GPT-4 里可能对应 "the"，在 LLaMA 里可能对应 "for"。Tokenizer 和模型是绑定的。

➡️ 转场

> 总结完了，最后留一个编程练习。我们之前实现了 BPE 的训练（学习合并规则），但缺少 encode——怎么用训练好的规则给新文本编码？

---

## 第七段：练习——实现 BPE encode（Cell 21, 23）

📍 运行 Cell 23（代码：BPE encode 实现 + TODO 补全 + 测试）

⏱ 时间分配：13 分钟

🎯 本段目标
- 学生理解 BPE 编码（encode）的算法：按合并规则顺序依次应用
- 补全 Cell 23 中的 TODO 代码
- 用训练好的合并规则验证编码结果

🗣 讲课话术

> Cell 21 的 Extra 部分提出了这个练习：我们之前写了 `get_stats` 和 `merge_vocab` 来训练 BPE，但缺少 `encode` 函数——给定一个新词和训练好的合并规则，怎么把它编码成 token 序列？
>
> 算法其实很简单：
> 1. 把输入词按字符拆开，加上 `</w>` 结束符
> 2. 按照训练时的合并顺序（很关键——必须按顺序！），依次查找并合并相邻的 token 对
> 3. 返回最终的 token 列表
>
> 大家看 Cell 23 的代码框架。`bpe_encode` 函数里有一个 TODO，需要大家补全。给 2 分钟先自己试。

### Hint 节奏

**0-2 分钟：** 自己尝试，不给提示。提示学生看函数上方的注释和算法描述。

**2 分钟第一个提示：**
> 核心逻辑：遍历 tokens 列表，如果当前位置的 token 和下一个 token 正好是 merge_pair，就把它们拼接成一个新 token。注意合并后要跳过两个位置（i += 2）。

**4 分钟关键代码：**
> ```python
> if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == merge_pair:
>     new_tokens.append(tokens[i] + tokens[i+1])
>     i += 2
> else:
>     new_tokens.append(tokens[i])
>     i += 1
> ```

### 常见错误

1. **忘记 `i += 2`**：合并后只跳了一步，导致重复合并或 index 错误
2. **合并顺序搞反**：没有按 merges 列表的顺序（优先级）来应用，而是随意合并
3. **忘记加 `</w>`**：初始 token 列表没有加结束符，导致合并规则匹配不上
4. **字符串拼接错误**：用 `' '.join` 而不是直接 `+`（tokens 列表里的元素不需要空格分隔）

### 验证标准

运行 Cell 23 后应看到：

训练好的 10 条合并规则（与 Cell 9 一致），然后是编码结果：
- `'low'` → `['low</w>']`（完全合并为一个 token）
- `'lower'` → `['low', 'e', 'r', '</w>']`（low 被合并，但 er 没有合并规则）
- `'newest'` → `['newest</w>']`（完全合并为一个 token）
- `'widest'` → `['wi', 'd', 'est</w>']`（wi 和 est</w> 被合并，d 单独保留）
- `'new'` → `['new', '</w>']`（new 被合并，但 new+</w> 没有合并规则）

重点指出：
- `'low'` 和 `'newest'` 是高频词，被完整合并成单个 token——这就是 BPE 的优势
- `'new'` 虽然 `new` 子词存在，但 `new</w>` 没有对应的合并规则，所以结束符保持独立
- `'widest'` 的 `wi` 来自第 10 次合并（频率只有 3），说明低频词拆分粒度更细

👀 输出要点
- 10 条合并规则打印（与 Cell 9 训练过程一致）
- 5 个测试词的编码结果
- 高频词（low, newest）被完整合并，低频词（lower, widest）有部分拆分

❓ 预判问题

Q: encode 的时间复杂度是多少？
A: 对于单个词，外层循环 M 次（合并规则数），内层扫描 token 列表（长度递减），大致是 O(M * L)，L 是初始字符数。实际生产中有更高效的实现（如 tiktoken 用 Rust 写的）。

Q: 为什么 `'new'` 没有被合并成 `'new</w>'`？
A: 看合并规则——第 9 条是 `low + </w> → low</w>`（频率 5），但没有 `new + </w>` 的合并规则。在训练语料中 "new" 从未作为独立词出现（只出现在 "newest" 中），所以 `new</w>` 没有被学到。

Q: 实际的 Tokenizer 是怎么处理整句话的？
A: 先用正则表达式把文本分成"预 token"（按空格、标点等边界分割），然后对每个预 token 分别运行 BPE encode。tiktoken 的正则在源码中可以看到，非常复杂。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，运行环境准备 | 0-4 |
| 5 | 三种分词策略对比 + 词表大小权衡 | 5-6 |
| 15 | BPE 算法理论 + 从头实现 | 7-9 |
| 35 | **休息** | -- |
| 40 | tiktoken 实战 + 多语言效率对比 | 10-13 |
| 50 | Tokenizer 经典 Bug（反转/数字/空格） | 14-17 |
| 65 | 词表可视化 + 总结 + 面试题 | 18-22 |
| 72 | 练习：BPE encode 实现 | 21, 23 |
| 85 | 结束 | -- |

---

## 附录 B：关键数据快速参考

### 核心公式

| 公式名称 | 数学表达 | 说明 |
|:---|:---|:---|
| BPE 合并规则 | best_pair = argmax freq(a,b) | 每轮合并频率最高的相邻对 |
| 压缩率 | bytes/token = 总字节数 / 总 token 数 | 越高表示 Tokenizer 越高效 |
| 词表参数量 | Embedding 参数 = V × d_model | V=100K, d=4096 → 4 亿参数 |
| Attention 成本 | O(T²)，T 与词表大小成反比 | 大词表 → 短序列 → 少计算 |
| BPE 训练复杂度 | O(n × V) | n=语料 token 数，V=目标词表 |

### 关键数值速查

| 数值 | 来源 | 含义 |
|:---|:---|:---|
| 37 / 5 / 8 tokens | Cell 6 | 字符级 / 词级 / 子词级对同一句话的分词结果 |
| 合并频率 9, 9, 9 | Cell 9 前 3 次合并 | (e,s), (es,t), (est,</w>) 都来自 "est" 后缀 |
| 合并频率 7, 7 | Cell 9 第 4-5 次合并 | (l,o), (lo,w) 来自 "low" |
| 13 tokens | Cell 11 | "Hello, 你好！I'm learning about tokenizers." 的 token 数 |
| 6.22 字符/token | Cell 13 English | 英文分词效率 |
| 0.94 字符/token | Cell 13 Chinese | 中文分词效率（不到英文的 1/6） |
| 3.89 字符/token | Cell 13 Code | 代码分词效率 |
| Token ID 15339 vs 24748 vs 9906 | Cell 17 | hello / (空格)hello / Hello 是完全不同的 token |

### 典型词表大小

| 模型 | 词表大小 | Tokenizer 类型 |
|:---|:---|:---|
| GPT-2 | 50,257 | Byte-level BPE |
| GPT-4 (cl100k_base) | ~100,000 | Byte-level BPE |
| BERT | 30,522 | WordPiece |
| LLaMA | 32,000 | SentencePiece (BPE) |
| LLaMA 3 | 128,000 | SentencePiece (BPE) |

---

## 附录 C：应急预案

### 场景 1：tiktoken 安装失败

**症状：** `ModuleNotFoundError: No module named 'tiktoken'`

**应对：**
1. 取消注释 Cell 12 的 `!pip install tiktoken` 并运行
2. 如果 pip 失败（网络问题），使用镜像源：`pip install tiktoken -i https://pypi.tuna.tsinghua.edu.cn/simple`
3. 最后手段：Cell 11 和 Cell 13 都有 `except ImportError` 分支，会打印模拟输出。口头讲解即可，不影响理解

### 场景 2：中文字体不显示

**症状：** Cell 9 和 Cell 19 的 matplotlib 图表中文显示为方框

**应对：**
1. Cell 4 已配置了 `plt.rcParams["font.sans-serif"]` 备选字体列表
2. 如果仍有问题，在 Cell 4 后插入：`plt.rcParams['font.sans-serif'] = ['DejaVu Sans']`（牺牲中文标签，保证图能看）
3. 口头补充中文标签含义："合并对"→ merged pair，"频率"→ frequency

### 场景 3：Cell 23 学生卡住（TODO 补全）

**应对：**
1. 源 notebook 已包含参考实现（TODO 注释后面就是答案代码）
2. 先给提示："用 while 循环遍历 tokens 列表，检查相邻两个是否等于 merge_pair"
3. 2 分钟后给出关键一行：`new_tokens.append(tokens[i] + tokens[i+1])`
4. 最多 4 分钟后展示完整答案，不要卡在这里太久

### 场景 4：时间不够

**可跳过的内容（按优先级）：**
1. Cell 19 词表可视化（口头说"英文效率是中文的 3 倍"即可）- 省 3 分钟
2. Cell 14 的 SolidGoldMagikarp 讲解（属于扩展知识）- 省 2 分钟
3. Cell 23 练习改为课后作业 - 省 10 分钟

**不可跳过的核心：**
- Cell 8-9：BPE 训练的完整过程（本章核心动手环节）
- Cell 11：tiktoken 逐 token 解码（理解 Tokenizer 工作方式的关键）
- Cell 15-17：至少演示一个 Bug（字符串反转最直观）

### 场景 5：学生提出超纲问题

**Q: Unigram 模型和 BPE 有什么区别？**
A: BPE 是自底向上合并（从字符开始不断合并），Unigram 是自顶向下剪枝（从一个大词表开始逐步删除低频 token）。SentencePiece 支持两种算法。记录下来，课后可以看 Karpathy 的视频深入了解。

**Q: 为什么不直接用 Unicode 编码？**
A: Unicode 有十几万个码点，直接用作词表太大了。字节级 BPE 把 Unicode 先拆成 UTF-8 字节（只有 256 种），再在字节基础上做 BPE，兼顾了覆盖率和词表大小。

**Q: Tokenizer 能在线更新吗？**
A: 通常不能——改了 Tokenizer 就意味着所有 Embedding 要重新训练。这就是为什么词表设计是一次性的重要决策。不过有些研究在探索增量扩展词表的方法。